# Boxplots de MD e S: C-NBI, VRF-NBI e NSGA-III (seed 6)

A figura reproduz a comparação distributiva das duas métricas, acrescentando somente a seed 6 do NSGA-III.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks').is_dir() and (p / 'configs').is_dir())
DATA_FILE = REPO / "outputs" / "01a00609-6550-79f2-8328-0714575f0c90" / "solucoes_4_metodos_cenario_aplicado.xlsx"
OUTPUT_DIR = REPO / "results" / "applied" / "figures_dissertation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METHOD_ORDER = ["C-NBI", "VRF-NBI", "NSGA-III"]
LABELS = ["C-NBI", "VRF-NBI", "NSGA-III\n(seed 6)"]
COLORS = ["#E68613", "#2878B5", "#3A923A"]

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 9.5,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [ ]:
df_all = pd.read_excel(DATA_FILE, sheet_name="Todas as soluções")
mask = (
    df_all["Método"].isin(["C-NBI", "VRF-NBI"])
    | ((df_all["Método"] == "NSGA-III") & (df_all["Semente"] == 6))
)
df = df_all.loc[mask, ["Método", "Semente", "MD", "S"]].copy()
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["MD", "S"])

summary = (
    df.groupby("Método", sort=False)[["MD", "S"]]
      .agg(["count", "mean", "std", "median", "min", "max"])
)
summary.to_csv(OUTPUT_DIR / "resumo_boxplots_md_s_cnbi_vrf_nsga_seed6.csv")
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.15, 3.35), constrained_layout=False)

meanprops = dict(marker="X", markerfacecolor="white", markeredgecolor="#222222",
                 markeredgewidth=0.9, markersize=5.6)
flierprops = dict(marker="o", markerfacecolor="none", markeredgecolor="#A6A6A6",
                  markeredgewidth=0.7, markersize=3.0, alpha=0.8)
medianprops = dict(color="#202020", linewidth=1.25)
whiskerprops = dict(color="#333333", linewidth=1.0)
capprops = dict(color="#333333", linewidth=1.0)

for ax, metric in zip(axes, ["MD", "S"]):
    values = [df.loc[df["Método"] == method, metric].to_numpy() for method in METHOD_ORDER]
    boxes = ax.boxplot(
        values, labels=LABELS, widths=0.56, patch_artist=True, showmeans=True,
        meanprops=meanprops, flierprops=flierprops, medianprops=medianprops,
        whiskerprops=whiskerprops, capprops=capprops,
    )
    for patch, color in zip(boxes["boxes"], COLORS):
        patch.set_facecolor(color)
        patch.set_edgecolor("#333333")
        patch.set_linewidth(1.0)
        patch.set_alpha(0.92)

    ax.set_ylabel(metric)
    ax.grid(axis="y", linestyle="--", linewidth=0.55, color="#D9D9D9", alpha=0.7)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_color("#333333")
        spine.set_linewidth(0.85)
    ax.tick_params(axis="x", pad=5)

axes[1].yaxis.set_label_coords(-0.11, 0.5)
fig.subplots_adjust(left=0.08, right=0.99, top=0.98, bottom=0.20, wspace=0.30)
png_path = OUTPUT_DIR / "fig_boxplots_md_s_cnbi_vrf_nsga_seed6.png"
pdf_path = OUTPUT_DIR / "fig_boxplots_md_s_cnbi_vrf_nsga_seed6.pdf"
fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
plt.show()
print(png_path)
print(pdf_path)